In [ ]:
########  distance matrix languages  ##########

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram, fcluster
from scipy.spatial.distance import squareform

# ============================================================
# 1. Load Excel file and select columns
# ============================================================
df = pd.read_excel(
    "input_language.xlsx"
)

lang_col = df.columns[0]
item_cols = df.columns[2:15]

data = df[item_cols].copy()
data.index = df[lang_col].astype(str)
data = data.dropna(how="all")

# ============================================================
# 2. MEAN Euclidean distance handling missing values
# ============================================================
def pairwise_mean_euclidean_with_missing(X: pd.DataFrame) -> pd.DataFrame:
    X_values = X.values.astype(float)
    n = X_values.shape[0]
    dist = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            xi = X_values[i, :]
            xj = X_values[j, :]

            mask = ~np.isnan(xi) & ~np.isnan(xj)

            if mask.sum() == 0:
                d = np.nan
            else:
                diff = xi[mask] - xj[mask]
                d = np.sqrt(np.mean(diff ** 2))

            dist[i, j] = d
            dist[j, i] = d

    np.fill_diagonal(dist, 0.0)
    return pd.DataFrame(dist, index=X.index, columns=X.index)

dist_matrix = pairwise_mean_euclidean_with_missing(data)

# ============================================================
# 3. Hierarchical clustering
# ============================================================
dist_for_clust = dist_matrix.copy()
max_dist = np.nanmax(dist_for_clust.values)
dist_for_clust = dist_for_clust.fillna(max_dist)

Z = linkage(squareform(dist_for_clust.values), method="average")

cut_distance = 1.6  # critical value

# Leaf order: this must govern EVERYTHING
leaf_order = leaves_list(Z)
ordered_labels = dist_matrix.index[leaf_order]
dist_ordered = dist_matrix.loc[ordered_labels, ordered_labels]

# Optional: print clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion="distance")
for cl in np.unique(cluster_labels):
    members = dist_matrix.index[cluster_labels == cl]
    print(f"Cluster {cl}: {list(members)}")

# ============================================================
# 4. Combined figure: dendrogram | labels | heatmap | colorbar
# ============================================================
cmap = LinearSegmentedColormap.from_list("white_to_darkred", ["white", "darkred"])
plot_data = np.ma.masked_invalid(dist_ordered.values)

vmin = 0
vmax = np.nanmax(dist_ordered.values)

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(
    1, 3,
    width_ratios=[3.0, 0.9, 10.0],
    wspace=0.01
)

ax_dendro = fig.add_subplot(gs[0, 0])
ax_names  = fig.add_subplot(gs[0, 1])
ax_hm     = fig.add_subplot(gs[0, 2])

# --- dendrogram: do NOT invert y-axis ---
dendrogram(
    Z,
    orientation="left",
    no_labels=True,
    color_threshold=cut_distance,
    ax=ax_dendro
)
ax_dendro.axvline(cut_distance, color="red", linestyle="--", linewidth=2)
ax_dendro.set_yticks([])
ax_dendro.set_xlabel("Distance")

# --- labels column: centered and WITHOUT invert_yaxis ---
ax_names.set_xlim(0, 1)
ax_names.set_ylim(-0.5, len(ordered_labels) - 0.5)
ax_names.axis("off")
for i, lab in enumerate(ordered_labels):
    ax_names.text(0.5, i, str(lab), va="center", ha="center", fontsize=10)

# --- heatmap: use origin="lower" so the order goes bottom-up like the dendrogram ---
im = ax_hm.imshow(
    plot_data,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest",
    aspect="auto",
    origin="lower"   # <--- KEY to avoid swapping top/bottom
)

ax_hm.set_yticks([])  # labels are in the central column
ax_hm.set_xticks(np.arange(len(ordered_labels)))
ax_hm.set_xticklabels(ordered_labels, rotation=75, ha="right", fontsize=10)
#ax_hm.set_title("Language Distance Matrix + Dendrogram", fontsize=14)

# ============================================================
# 5. Colorbar as wide as one heatmap cell
# ============================================================
fig.canvas.draw()
pos = ax_hm.get_position()
ncols = dist_ordered.shape[1]
cell_width_fig = pos.width / ncols

gap = cell_width_fig
cb_width = cell_width_fig

cb_left = pos.x1 + gap
cb_bottom = pos.y0
cb_height = pos.height

cax = fig.add_axes([cb_left, cb_bottom, cb_width, cb_height])
fig.colorbar(im, cax=cax, orientation="vertical")

fig.savefig("distance_matrix_languages.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
##########  correlation matrix languages ###############

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import squareform

# ============================================================
# 1. Load Excel file and select columns
# ============================================================
df = pd.read_excel(
    "input_language.xlsx"
)

lang_col = df.columns[0]
item_cols = df.columns[2:15]

data = df[item_cols].copy()
data.index = df[lang_col].astype(str)
data = data.dropna(how="all")

# ============================================================
# 2. Mean Euclidean distance with missing values
#    (used ONLY to obtain the same ordering as the dendrogram
#     from the already constructed distance matrix)
# ============================================================
def pairwise_mean_euclidean_with_missing(X: pd.DataFrame) -> pd.DataFrame:
    X_values = X.values.astype(float)
    n = X_values.shape[0]
    dist = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            xi = X_values[i, :]
            xj = X_values[j, :]

            mask = ~np.isnan(xi) & ~np.isnan(xj)

            if mask.sum() == 0:
                d = np.nan
            else:
                diff = xi[mask] - xj[mask]
                d = np.sqrt(np.mean(diff ** 2))

            dist[i, j] = d
            dist[j, i] = d

    np.fill_diagonal(dist, 0.0)
    return pd.DataFrame(dist, index=X.index, columns=X.index)

dist_matrix = pairwise_mean_euclidean_with_missing(data)

# ============================================================
# 3. Hierarchical clustering to obtain the SAME ordering
# ============================================================
dist_for_clust = dist_matrix.copy()
max_dist = np.nanmax(dist_for_clust.values)
dist_for_clust = dist_for_clust.fillna(max_dist)

Z = linkage(squareform(dist_for_clust.values), method="average")

cut_distance = 1.6

leaf_order = leaves_list(Z)
ordered_labels = dist_matrix.index[leaf_order]

# ============================================================
# 4. Correlation matrix with missing value handling
# ============================================================
def pairwise_correlation_with_missing(X: pd.DataFrame) -> pd.DataFrame:
    X_values = X.values.astype(float)
    n = X_values.shape[0]
    corr = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):
            xi = X_values[i, :]
            xj = X_values[j, :]

            mask = ~np.isnan(xi) & ~np.isnan(xj)

            if mask.sum() < 2:
                c = np.nan
            else:
                c = np.corrcoef(xi[mask], xj[mask])[0, 1]

            corr[i, j] = c
            corr[j, i] = c

    np.fill_diagonal(corr, 1.0)
    return pd.DataFrame(corr, index=X.index, columns=X.index)

corr_matrix = pairwise_correlation_with_missing(data)

# Reorder the correlation matrix using the same ordering as the distance matrix
corr_ordered = corr_matrix.loc[ordered_labels, ordered_labels]

# ============================================================
# 5. Combined figure: dendrogram | labels | correlation heatmap
# ============================================================
corr_plot_data = np.ma.masked_invalid(corr_ordered.values)

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(
    1, 3,
    width_ratios=[3.0, 0.9, 10.0],
    wspace=0.01
)

ax_dendro = fig.add_subplot(gs[0, 0])
ax_names  = fig.add_subplot(gs[0, 1])
ax_hm     = fig.add_subplot(gs[0, 2])

# --- dendrogram on the left ---
dendrogram(
    Z,
    orientation="left",
    no_labels=True,
    color_threshold=cut_distance,
    ax=ax_dendro
)

ax_dendro.axvline(cut_distance, color="red", linestyle="--", linewidth=2)
ax_dendro.set_yticks([])
ax_dendro.set_xlabel("Distance")

# --- column containing language names ---
ax_names.set_xlim(0, 1)
ax_names.set_ylim(-0.5, len(ordered_labels) - 0.5)
ax_names.axis("off")

for i, lab in enumerate(ordered_labels):
    ax_names.text(0.5, i, str(lab), va="center", ha="center", fontsize=10)

# --- correlation heatmap ---
im = ax_hm.imshow(
    corr_plot_data,
    cmap=plt.cm.seismic,
    vmin=-1,
    vmax=1,
    interpolation="nearest",
    aspect="auto",
    origin="lower"   # crucial for alignment with the dendrogram
)

ax_hm.set_yticks([])  # labels are shown in the central column
ax_hm.set_xticks(np.arange(len(ordered_labels)))
ax_hm.set_xticklabels(ordered_labels, rotation=75, ha="right", fontsize=10)
# ax_hm.set_title("Correlation Matrix + Dendrogram", fontsize=14)

# ============================================================
# 6. Colorbar with the same width as one heatmap cell
# ============================================================
fig.canvas.draw()
pos = ax_hm.get_position()
ncols = corr_ordered.shape[1]
cell_width_fig = pos.width / ncols

gap = cell_width_fig
cb_width = cell_width_fig

cb_left = pos.x1 + gap
cb_bottom = pos.y0
cb_height = pos.height

cax = fig.add_axes([cb_left, cb_bottom, cb_width, cb_height])
fig.colorbar(im, cax=cax, orientation="vertical")

# ============================================================
# 7. Save figure with a different filename
# ============================================================
fig.savefig("correlation_matrix_languages.png", dpi=300, bbox_inches="tight")
plt.show()